# STAT 764 · Meeting 22 — Why is it blank?

**Thursday, November 19**

Every dataset this term had blanks, and every model had to do something with them: drop the row, fill in a number,
or treat "blank" as a category. Each of those is a guess about **why** the value is blank, usually made without
saying so. Today makes the guess out loud.

| the question | today |
|---|---|
| Can a blank carry information? | Section 1 · the A1C test nobody ordered |
| Why is it blank, and does the reason matter? | Section 2 · make blanks three ways, and see what each fix does |
| How do you fill a blank honestly? | Section 3 · inside the pipeline, and how sure the fill is |

Then the **capstone data clinic**: the studio runs the same checks on your own team's data.

The page `missing.html` in this folder has the same ideas with pictures. Open it in a web browser.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

In [ ]:
import numpy as np
import pandas as pd
from stat764 import diabetes, load

from sklearn.experimental import enable_iterative_imputer  # noqa: F401 (switches IterativeImputer on)
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# The readmission data from Meetings 9-20: one row per hospital stay, without the stays that ended in death or hospice.
enc = diabetes()
enc = enc[~enc["discharge_disposition_id"].isin([11, 13, 14, 19, 20, 21])].copy()
enc["came_back"] = (enc["readmitted"] == "<30").astype(int)     # back in the hospital within 30 days: 1, else 0
print(f"{len(enc):,} hospital stays · {enc['came_back'].mean():.1%} came back within 30 days")

## 1. The A1C test nobody ordered

The **A1C** is a blood test of average blood sugar over the past three months. In this data it is blank for most
stays, because nobody ordered it. Since Meeting 9 the models have treated that blank as its own category (the
encoder called it `nan`): Meeting 15 said not being tested "can itself say something about a patient." Does it?

In [ ]:
# Name the blank for what it is: the test was not done.
a1c = enc["A1Cresult"].fillna("not measured")
print(f"A1C not measured for {a1c.eq('not measured').mean():.1%} of stays\n")

# How often each group came back. ">7" and ">8" are high results (poorly controlled diabetes); "Norm" is normal.
print(enc.groupby(a1c)["came_back"].agg(stays="size", came_back="mean").round(3).to_string())

The stays with **no test** came back more often than the stays with **any** result, normal or high. What the test
found barely matters; whether anyone ordered it does. So the blank is information, and a model that throws it away
(or fills it with a typical result) loses it.

Who gets tested? If testing depended on nothing, the tested stays would look like everyone else.

In [ ]:
tested = enc["A1Cresult"].notna()
main_diabetes = enc["diag_1"].astype(str).str.startswith("250")   # diagnosis code 250.xx: diabetes is the main reason

# Share of stays tested: by age band, and by whether diabetes was the main reason for the stay.
print("share tested, by age:")
print(tested.groupby(enc["age"]).mean().round(3).to_string())
print(f"\nshare tested: diabetes the main reason {tested[main_diabetes].mean():.1%} · otherwise {tested[~main_diabetes].mean():.1%}")

# What dropping the untested stays would leave.
print(f"\nkeep only the tested stays: {tested.sum():,} of {len(enc):,} ({tested.mean():.1%}),"
      f" came back {enc.loc[tested, 'came_back'].mean():.1%} (all stays: {enc['came_back'].mean():.1%})")

Testing depends on things the data **does** record: the young are tested far more often than the old, and stays
for diabetes itself twice as often as the rest. Drop the untested stays and the model is about a different, younger,
diabetes-first group of patients, with a lower rate of coming back: not the hospital's patients.

## 2. Why is it blank? Three reasons

| the reason | in words | example |
|---|---|---|
| **MCAR**, missing completely at random | the blank has nothing to do with anything | a lab machine was down that week |
| **MAR**, missing at random | whether it is blank depends on other columns you **have**, not on the hidden value | the A1C, if among patients with the same diagnosis and age, who got tested has nothing to do with their blood sugar |
| **MNAR**, missing not at random | the blank depends on the **hidden value itself**, or on something the data do not record | a county does not report its rate when the rate is bad |

(The names are old and confusing: "missing at random" does **not** mean random. Read MAR as "random only within
groups you can see".)

You cannot check which reason you have when the values are truly missing. So make the blanks yourself, on data
where you know the truth: the **unemployment rate** of the 2,954 counties from Meeting 21. Hide 30% of them three
ways, then try the two common fixes: **drop** the blank counties, or **fill** each blank from the county's other 38
measures (`IterativeImputer`: it predicts the blank column from the others with a regression, and repeats).

In [ ]:
# Meeting 21's county data: 39 rates for each county, from County Health Rankings 2025.
raw = load("county_health_2025.csv", low_memory=False, skiprows=[1])          # row 2 holds variable codes: skip it
raw = raw[pd.to_numeric(raw["County FIPS Code"], errors="coerce") != 0]     # FIPS 0 rows are states, not counties
MEASURES = [
    "Poor or Fair Health", "Poor Physical Health Days", "Poor Mental Health Days", "Frequent Physical Distress",
    "Frequent Mental Distress", "Diabetes Prevalence", "Adult Obesity", "Adult Smoking", "Physical Inactivity",
    "Excessive Drinking", "Insufficient Sleep", "Food Insecurity", "% Disability: Functional Limitations",
    "Uninsured", "Uninsured Adults", "Uninsured Children", "Flu Vaccinations", "Mammography Screening",
    "Preventable Hospital Stays", "High School Completion", "Some College", "Unemployment", "Children in Poverty",
    "Median Household Income", "Income Inequality", "Children in Single-Parent Households", "Social Associations",
    "Broadband Access", "Homeownership", "Severe Housing Problems", "Severe Housing Cost Burden",
    "Air Pollution: Particulate Matter", "Driving Alone to Work", "Long Commute - Driving Alone", "% Rural",
    "% 65 and Older", "% Below 18 Years of Age", "% Non-Hispanic White", "% Hispanic"]
counties = raw[["State Abbreviation", "Name"]].rename(columns={"State Abbreviation": "state", "Name": "county"})
for m in MEASURES + ["Life Expectancy"]:
    counties[m] = pd.to_numeric(raw[f"{m} raw value"], errors="coerce")
complete = counties.dropna(subset=MEASURES + ["Life Expectancy"]).reset_index(drop=True)

X = complete[MEASURES].copy()
X["Unemployment"] = X["Unemployment"] * 100        # as a percent: 3.6 means 3.6% of the labor force
truth = X["Unemployment"].mean()
print(f"{len(X):,} counties · true average unemployment {truth:.2f}%")


def chance(z):
    """Each county's chance of being blanked: rises with z, and averages 30% over the counties."""
    shift = 0.0
    for _ in range(50):                            # nudge the curve up or down until the average chance is 30%
        shift += 0.30 - (1 / (1 + np.exp(-(z + shift)))).mean()
    return 1 / (1 + np.exp(-(z + shift)))


def standardized(s):
    return (s - s.mean()) / s.std()


# Three ways to make blanks in the unemployment column.
how = {
    "MCAR: at random": np.full(len(X), 0.30),
    "MAR: more where child poverty is high": chance(2 * standardized(X["Children in Poverty"])),
    "MNAR: more where unemployment is high": chance(2 * standardized(X["Unemployment"])),
}
rng = np.random.default_rng(764)
col = MEASURES.index("Unemployment")
rows, blanked = [], {}
for name, p in how.items():
    blank = rng.random(len(X)) < p                 # True for the counties whose unemployment we hide
    X_blank = X.copy()
    X_blank.loc[blank, "Unemployment"] = np.nan
    filled = IterativeImputer(random_state=0).fit_transform(X_blank)[:, col]   # fill from the other 38 measures
    blanked[name] = (blank, X_blank)
    rows.append({"blanks made": name, "counties blanked": blank.sum(),
                 "drop the blanks": X_blank["Unemployment"].mean(),        # average over the counties left
                 "fill from the others": filled.mean(),                    # average over all counties, blanks filled
                 "child poverty %, blanked": 100 * X.loc[blank, "Children in Poverty"].mean(),
                 "child poverty %, the rest": 100 * X.loc[~blank, "Children in Poverty"].mean()})
print(f"\naverage unemployment, %: the truth is {truth:.2f}")
print(pd.DataFrame(rows).set_index("blanks made").round(2).to_string())

Read the rows against the truth, 3.59%:

- **MCAR:** both fixes land on the truth. Random blanks cost data, not accuracy. (For an average, that is the only
  case where dropping is safe; for a model, dropping also changes who the model is about, as Section 1 showed.)
- **MAR:** dropping is wrong (3.37): the blanked counties were the poorer ones, with more unemployment. Filling from
  the other measures gets it right (3.60), because child poverty, which caused the blanks, is in the data.
- **MNAR:** both are wrong. Filling helps (3.31 against 3.15) only as far as the other measures know a county's
  unemployment; what they do not know, nothing in the data can recover.

Now the last two columns. Under MAR **and** under MNAR the blanked counties are poorer than the rest (26% and 23% of
children in poverty, against 16% and 17%). So the data
can show you that blanks are **not** random, but not whether they are MAR or MNAR: that takes knowing how the data
were collected. Ask whoever made it.

**One more kind: censored values.** Sometimes a value is not missing but cut off: you know it lies beyond a limit,
not where (a lab result "< 0.5"; a patient with "no readmission yet" when the data end). It is a cousin of MNAR, but
you know which side of the limit it is on. Don't fill it with a typical value and don't drop it: keep the limit and a
0/1 "cut off" column, or use a method built for it (survival analysis, Tobit regression). In a capstone, a training
row whose outcome had not happened by your cutoff date is censored.

**What to do, by kind of blank:**

| kind | for an average or another estimate | for a prediction model |
|---|---|---|
| MCAR | almost anything works: dropping only costs precision | fill inside the pipeline; the median is fine |
| MAR | fill from the columns that explain the blanks (3.60); don't drop (3.37) | fill from the other columns, inside the pipeline: `IterativeImputer` (Section 3) or `KNNImputer`; or a model that takes blanks as they are (`HistGradientBoosting`) |
| MNAR | nothing in the data fixes it (best: 3.31): find out why, or show how the answer moves | fill, plus a 0/1 "was blank" column: the blank itself often predicts |
| censored | never fill with a typical value, never drop: keep the limit and a 0/1 "cut off" column | same for a feature; for the outcome, train only on rows known by your cutoff |

For prediction, one default covers the first three: fill inside the pipeline and add the "was blank" column
(Section 3). That matters because the data cannot tell MAR from MNAR. The real risk is the blanks' cause changing
after training.

## 3. Filling in honestly

**A fill is learned, so it goes inside the pipeline**, like the scaler and PCA: the medians, or the regressions an
`IterativeImputer` uses, come from the training rows only. In a pipeline, the model below predicts every county,
blanks and all; nothing is dropped.

In [ ]:
# The MAR blanks, inside a pipeline that fills, scales and fits, scored by state as in Meeting 21.
blank, X_mar = blanked["MAR: more where child poverty is high"]
y, state = complete["Life Expectancy"], complete["state"]
fill_then_fit = make_pipeline(IterativeImputer(random_state=0), StandardScaler(), LinearRegression())
rmse = -cross_val_score(fill_then_fit, X_mar, y, groups=state, cv=GroupKFold(n_splits=5),
                        scoring="neg_root_mean_squared_error").mean()
print(f"{blank.sum()} counties with unemployment blank; every county predicted; RMSE {rmse:.2f} years")

Two more tools. `SimpleImputer(strategy="median", add_indicator=True)` fills each blank with the training median
**and** adds a 0/1 column saying "this was blank": the number-column version of the A1C's "not measured" category.

**A fill is a guess.** Every fill so far puts one number in each blank, and from then on the model treats it exactly
like a measured value. To see how sure the guess is, ask for plausible values instead:
`IterativeImputer(sample_posterior=True)` adds a random amount of its own uncertainty to each prediction, so five
seeds give five plausible fills. Where they agree, the guess is solid; where they scatter, it is not. The MAR blanks
are the place to look, because we know the hidden values.

In [ ]:
# Five different draws of the MAR blanks: each fill is one plausible value, not the truth.
draws = [IterativeImputer(sample_posterior=True, random_state=s).fit_transform(X_mar)[:, col] for s in range(5)]
best = IterativeImputer(random_state=0).fit_transform(X_mar)[:, col]          # the single best guess, as in Section 2
for i in np.flatnonzero(blank)[:3]:                                           # three of the blanked counties
    print(f"{complete.loc[i, 'county']}, {complete.loc[i, 'state']}: true {X['Unemployment'].iloc[i]:.2f}"
          f" · single fill {best[i]:.2f} · five draws {', '.join(f'{d[i]:.2f}' for d in draws)}")
spread = np.std(np.vstack(draws)[:, blank], axis=0)                           # spread of the 5 draws, per county
print(f"\ntypical spread of the draws for one county: ±{np.median(spread):.2f} points (unemployment varies by"
      f" ±{X['Unemployment'].std():.2f} across counties)")
print("average over all counties, each draw:", ", ".join(f"{d.mean():.2f}" for d in draws))

A single fill looks as sure as a measured value, but one county's draws spread over a point or more (Autauga
County: 1.65 to 5.05). Averages over many counties barely move (every draw gives about 3.6); one county's number
does. Fitting the
model once on each of several draws, and reporting how much the answer moves, is **multiple imputation**, the usual
way to carry that uncertainty forward.

**Before filling any blank, ask:**

1. **Why is it blank?** Does "blank" mean "none" or "not applicable"? Was it unknown at the moment of prediction?
   MCAR, MAR or MNAR? Who would know?
2. **Does the blank say something?** Compare the outcome when blank and when filled. If it differs, keep the blank
   as information: a category, or an indicator column.
3. **Who does dropping remove?** If you drop rows, say who they are, because the model will not be about them.

## Studio · the capstone data clinic

**Run these checks on your own team's capstone data.** Work with your team and compare as you go, but each of you
uploads your own notebook. If your team's data will not load on your laptop today, use the readmission data above
(`df = enc`, `OUTCOME = "came_back"`) and say so in your answers.

| | what you practice | about |
|---|---|---|
| **Question 1** | counting the blanks, including the ones in disguise | 10 minutes |
| **Question 2** | asking whether a blank says something about the outcome | 15 minutes |
| **Question 3** | a missing-data plan for your capstone | 15 minutes |

**All three questions count.** If you do not finish in class, finish at home: the notebook is due 24 hours after
class.

**How to work.**

1. Run the notebook from the top (**Run All**) first: it takes under a minute.
2. In the first studio cell, load the table your team models from (copy the loading code from your capstone
   notebook) into `df`, and set `OUTCOME` to the name of your 0/1 outcome column.
3. Before questions 1 and 2, **write your guess** as a comment in the cell, and keep it when the answer disagrees.
   The Compare uses it.
4. Each question ends with sentences to answer. Write them in a comment or a markdown cell under its output.
5. Save and upload to Canvas under "Studio notebook — Day 22" by **Friday, November 20, 3:40 pm**.

**Stuck?**

| you see | what to do |
|---|---|
| `FileNotFoundError` | The path in your loading code points at your capstone folder; copy it from your capstone notebook, or use the full path. |
| `KeyError` for your outcome | `OUTCOME` must match a column name exactly: `print(df.columns.tolist())`. |
| Your outcome is not 0/1 | Make it one first, as your capstone notebook does, e.g. `df["late"] = (df["ArrDelay"] >= 15).astype(int)` (use your file's column name). Careful: that turns a **blank** delay (a cancelled flight) into 0, on time. Count those blanks first. |
| Your outcome itself has blanks | That is a finding: say which rows and why (a cancelled flight has no arrival delay). Use `df.dropna(subset=[OUTCOME])` for question 2, and count what you dropped. |
| No column has a blank | Look for disguised ones (question 1b). If your team already dropped rows, run questions 1 and 2 on the file **before** those rows were dropped. |
| Question 2 prints one-row tables | Those columns are blank in every row (some flight files have dozens); use only the partly blank ones. |

### 1. Count the blanks

- **(a)** For every column with a blank: how many, and what share of rows. Largest share first:
  `df.isna().sum()`, then `df.isna().mean()`, keeping the columns above 0.
- **(b)** Blanks in disguise. Count, in each text column, the values that mean "blank" without being one:
  `df.select_dtypes(exclude="number").isin(["?", "Undefined", "Unknown", "unknown", "-"]).sum()`. And look
  for impossible numbers your data might use for "unknown" (a 0 where 0 cannot happen, a 999). Note: when pandas read
  the file it already turned the words NULL, NA and N/A, and empty cells, into blanks, so some of (a)'s blanks were
  words in the file. Ask what they meant: the hotel file's NULL in `agent` means "no agent".
- **(c)** If your team has already dropped rows (duplicates, cancelled flights, bookings with no guests), say how
  many and why.

Guess first: which column of your data has the most blanks?

**Answer in two or three sentences:** which columns have blanks, and for each of the biggest, what does "blank"
mean there? Is it really missing, or does it mean "none" or "does not apply"?

### 2. Does a blank say something?

- **(a)** For the three **partly** blank columns (blank in some rows, not all) with the most blanks (fewer if your
  data has fewer), the outcome rate when the column is blank and when it is filled, with how many rows each. One
  table per column is fine:

  ```python
  share = df.isna().mean()
  for c in share[(share > 0) & (share < 1)].nlargest(3).index:
      print(df.groupby(df[c].isna())[OUTCOME].agg(["size", "mean"]), "\n")
  ```
- **(b)** For the column whose blank differs most, compare blank and filled rows on one other column that might
  explain it (as Section 1 compared tested and untested stays by age).

Guess first: will any blank change the outcome rate by more than 5 points?

**Answer in two or three sentences:** which blank says the most about the outcome, and by how much? From (b), are
the blanks random? If not, what reason the data do not record could make them MNAR, and who would know?

### 3. Your team's missing-data plan

Write a markdown table with one row for each of your three columns from question 2:

| column | why it is blank | what we will do | who that affects |
|---|---|---|---|
| | | | |

- **why it is blank:** means "none" · unknown at the moment of prediction · the outcome did not happen as defined ·
  MCAR · MAR · MNAR · censored (known only to be beyond a limit) · don't know yet (and who you would ask).
- **what we will do:** its own category · fill inside the pipeline with an indicator column · drop the column ·
  drop the rows.
- **who that affects:** the rows changed or removed, with a count.

**Answer in two sentences:** which row of your plan are you least sure of, and what would settle it?

In [ ]:
# YOUR CODE HERE
# df = ...          your team's modeling table
# OUTCOME = "..."   your 0/1 outcome column

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

In [ ]:
# YOUR CODE HERE
# my guess, before computing:

### Question 3 · your team's missing-data plan

*(double-click to edit)*

| column | why it is blank | what we will do | who that affects |
|---|---|---|---|
| | | | |
| | | | |
| | | | |

**Least sure of, and what would settle it:**

## Compare — at the start of next class

We compare answers at the start of next class (Tuesday, November 24), built from the notebooks you upload. So leave
these in your notebook, in words or as printed numbers:

1. Your guesses, written before you computed, next to what you found.
2. Question 2: your table of outcome rates, blank against filled.
3. Question 3: your plan.

## Exit ticket

> A teammate fills every blank in your data with its column's median, on the whole dataset, before splitting it into
> training and test. In two or three sentences: name two things that could go wrong.

**Answer in the cell below, then save this notebook and upload it to Canvas under "Studio notebook — Day 22". Your
two studio points need this answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ESL §9.6** — this week's reading, short: missing predictor values, and the case for treating "missing" as its
own category in trees.

📗 **Stef van Buuren, *Flexible Imputation of Missing Data*** (free online) — the standard practical book on multiple
imputation, with MCAR, MAR and MNAR in its first chapter.

📗 **Strack et al. (2014), "Impact of HbA1c measurement on hospital readmission rates"** — the paper this
readmission data was built for. Its question is Section 1's: does whether the A1C was **measured** relate to coming
back?

🎓 **ISLP §12.3, matrix completion** — filling in a whole table's blanks at once with principal components, Meeting
21's tool; the idea behind recommender systems.

🎓 **Sensitivity analysis for MNAR** — when you suspect MNAR, assume the blanks are worse (or better) than the fill
by a set amount, and see whether your conclusion survives.